# 06 — CP1 benchmark: B1 profile nguồn

B1 đã có kết quả: inventory, sampling/gaps, proxy nhiễu và nguồn di chuyển.
Giữ setting và outputs của lượt đã chạy; RUN_B1_SCAN vẫn mang giá trị bạn đã lưu.

**B2 được tách sang `07_cp1_b2_synthetic.ipynb`.** Mở notebook 07 và chạy từ đầu;
không cần chạy lại các cell B1 trong notebook này.


In [ ]:
from pathlib import Path
import json, time, sys, hashlib, ctypes, subprocess, csv, gzip
from contextlib import ExitStack
import numpy as np
import pandas as pd
from IPython.display import display
from geolife_inference.geo import haversine_m
ROOT = Path.cwd().resolve()
if not (ROOT / 'CHECKPOINT.md').exists():
    ROOT = ROOT.parent
assert (ROOT / 'configs/baseline.toml').exists(), 'Mở notebook từ repo hoặc notebooks/'
OUT = ROOT / 'artifacts/benchmark/b1'
PROFILE_OUT = OUT / 'profiles_v1'
CLEANED = ROOT / 'artifacts/preprocessing/cleaned_points.csv.gz'
RUN_B1_SCAN = True
CHUNKSIZE = 100_000
WALL_BUDGET_SECONDS = 12 * 3600
COMPUTE_USED_BEFORE_SECONDS = 0.0  # Ghi compute benchmark mới thực đã dùng; 0 chỉ khi chưa có job mới.
OUT.mkdir(parents=True, exist_ok=True)
def readj(path):
    return json.loads((ROOT / path).read_text(encoding='utf-8'))
def writej(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
prep = readj('artifacts/preprocessing/audit.json')
expected_users = sorted(str(x) for x in prep['expected_users'])
assert len(expected_users) == 182
print('Kernel:', sys.executable)
print('Full scan enabled:', RUN_B1_SCAN)


## 1. Inventory đã lưu
Đọc audits/CSV nhỏ đủ mọi user; đây không phải kết quả scan profile mới. Các file nhãn được
kiểm tra cấu trúc/time intervals theo đồng hồ gốc chưa gán timezone. Số user có file nhãn không
đồng nghĩa đủ nhãn ô tô/bus/tàu hoặc có mẫu số cho (c). Việc xác minh UTC/local phải làm riêng.


In [ ]:
users = pd.DataFrame(prep['per_user'])
users['user_id'] = users.user_id.astype(str).str.zfill(3)
coverage = pd.read_csv(ROOT / 'artifacts/inference/coverage.csv', dtype={'user_id':str})
stays = pd.read_csv(ROOT / 'artifacts/stays/per_user.csv', dtype={'user_id':str})
users = users.merge(coverage.drop(columns=['processed_points','processed_trajectories']), on='user_id', validate='one_to_one')
users = users.merge(stays[['user_id','stay_count','stay_minutes']], on='user_id', validate='one_to_one')
label_rows = []
for p in sorted((ROOT / 'data/Geolife Trajectories 1.3/Data').glob('*/labels.txt')):
    labels = pd.read_csv(p, sep='\t')
    starts = pd.to_datetime(labels.iloc[:,0], format='%Y/%m/%d %H:%M:%S', errors='coerce')
    ends = pd.to_datetime(labels.iloc[:,1], format='%Y/%m/%d %H:%M:%S', errors='coerce')
    modes = labels.iloc[:,2].astype(str).str.strip().str.lower()
    bad = starts.isna() | ends.isna() | ends.le(starts)
    good = sorted(zip(starts[~bad], ends[~bad]))
    overlap_count = 0
    last_end = None
    for left,right in good:
        if last_end is not None and left < last_end: overlap_count += 1
        last_end = right if last_end is None else max(last_end,right)
    label_rows.append({'user_id':p.parent.name, 'label_rows':len(labels),
        'modes':';'.join(sorted(modes.unique())), 'invalid_label_intervals':int(bad.sum()),
        'overlapping_label_intervals':overlap_count,
        'motorized_label_rows':int((~bad & modes.isin(['car','taxi','bus','train','subway'])).sum()),
        'label_time_basis':'unverified_source_clock'})
label_inventory = pd.DataFrame(label_rows)
users = users.merge(label_inventory, on='user_id', how='left', validate='one_to_one')
users['label_rows'] = users.label_rows.fillna(0).astype(int)
users['raw_share_pct'] = 100 * users.input_points / users.input_points.sum()
users = users.sort_values(['input_points','user_id'], ascending=[False,True])
users['cumulative_raw_share_pct'] = users.raw_share_pct.cumsum()
assert set(users.user_id) == set(expected_users)
assert users.input_points.sum() == 24_876_978 and users.output_points.sum() == 24_177_772
users.to_csv(OUT / 'saved_output_user_inventory.csv', index=False)
label_inventory.to_csv(OUT / 'label_file_inventory.csv', index=False)
cli = readj('artifacts/pipeline/dbscan-v1/run-audit.json')
comparison = readj('artifacts/pipeline/dbscan-v1/notebook-comparison.json')
heuristic_audit_path = ROOT / 'artifacts/pipeline/heuristic-v1/run-audit.json'
manifest = {'status':'inventory_completed_scan_pending', 'scope':'all_182_users_saved_outputs',
    'users':len(users), 'raw_points':int(users.input_points.sum()), 'cleaned_points':int(users.output_points.sum()),
    'label_files':len(label_inventory), 'preprocessing_elapsed_seconds':prep.get('elapsed_seconds'),
    'dataset_fingerprint':prep['dataset_fingerprint'], 'cleaned_bytes':CLEANED.stat().st_size,
    'cleaned_mtime_ns':CLEANED.stat().st_mtime_ns,
    'baseline_config_sha256':hashlib.sha256((ROOT/'configs/baseline.toml').read_bytes()).hexdigest(),
    'dbscan_cli_status':cli['status'], 'dbscan_difference_count':comparison['difference_count'],
    'heuristic_cli_audit_exists':heuristic_audit_path.exists(),
    'runtime_note':'Saved CLI audit lacks per-user runtime and RAM; NB02 elapsed is not detector cost.',
    'pending':['sampling/gaps/noise/source profile scan','raw duplicate/altitude audit',
               'label timezone and contradictory modes verification','IANA timezone inventory','heuristic CLI parity/runtime']}
manifest['git_head'] = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
core_hash = hashlib.sha256()
for module in sorted((ROOT/'src/geolife_inference').glob('*.py')):
    core_hash.update(module.name.encode()); core_hash.update(module.read_bytes())
manifest['core_python_sha256'] = core_hash.hexdigest()
manifest['profile_output'] = str(PROFILE_OUT.relative_to(ROOT))
manifest['notebook_prepared'] = 'notebooks/06_cp1_benchmark.ipynb'
writej(OUT / 'inventory_manifest.json', manifest)
display(users[['user_id','input_points','raw_share_pct','cumulative_raw_share_pct','observed_days','stay_count','label_rows']].head(10))
print('Top 5 raw share:', users.head(5).raw_share_pct.sum())
print('Label files:', len(label_inventory))


## 2. Định nghĩa profile
- Đọc **mọi cleaned points**, giữ ranh giới user/trajectory/observation segment và thời gian UTC.
- Observed interval: cùng trajectory/segment, gap dương ≤5 phút; union qua files để không cộng đôi.
- Sampling blocks giữ nguyên thứ tự các dt, kể cả gaps và segment breaks, bằng run-length encoding
  theo từng trajectory. Generator có thể rút block của **chính user đó**; không chỉ dùng dt trung bình.
- Proxy nhiễu: residual của point giữa so với nội suy hai hàng xóm, chỉ khi hai pairs có dt ≤30 giây,
  speed ≥2 m/s, lệch speed ≤25% và đổi hướng ≤20°. Đây là điều kiện chọn đoạn tương đối đều,
  không dùng radius/anchor stay. Residual chứa cả cong đường/gia tốc, **không phải GPS error truth**.
- Nguồn di chuyển: trong mỗi observed run, lấy các đoạn không chồng nhau dài ít nhất 10 phút;
  ≥90% pair speed ≥2 m/s, không run speed <0,5 m/s dài >30 giây. Giữ toàn bộ đoạn đạt, không lấy
  mẫu points hoặc chạy detector để chọn. Nhãn phương tiện chưa xác minh đồng hồ nên scanner
  dùng nhánh speed; không tự khẳng định nền thật không có stop. Generator B2 sẽ kiểm soát latent path.
- Unobserved-support hours là thời gian chưa biết trong support trajectory, **không phải dwell**.

Output namespace `profiles_v1` khác thiết kế cũ. Scanner không ghi đè kết quả đã tồn tại; xem audit
trước khi chủ động chạy lại. Budget kiểm tra tại checkpoint, không phải giới hạn RAM cứng của hệ điều hành.


In [ ]:
def merge_intervals(intervals):
    result = []
    for s,e in sorted(intervals):
        if e <= s: continue
        if result and s <= result[-1][1]:
            result[-1][1] = max(e, result[-1][1])
        else: result.append([int(s),int(e)])
    return np.asarray(result, dtype=np.int64).reshape(-1,2)

class CoverageArea:
    def __init__(self, intervals):
        self.iv = merge_intervals(intervals)
        self.starts = self.iv[:,0]; self.ends = self.iv[:,1]
        self.cum = np.r_[0, np.cumsum(self.ends-self.starts)]
    def at(self, x):
        x = np.asarray(x, dtype=np.int64)
        if not len(self.iv): return np.zeros_like(x)
        n = np.searchsorted(self.ends, x, side='right')
        i = np.minimum(n, len(self.iv)-1)
        return self.cum[n] + np.where(n<len(self.iv), np.maximum(0,x-self.starts[i]),0)
    def between(self, start, end): return self.at(end)-self.at(start)

def process_memory():
    class PMC(ctypes.Structure):
        _fields_ = [('cb',ctypes.c_ulong),('faults',ctypes.c_ulong)] + [(k,ctypes.c_size_t) for k in
            ['peak_ws','ws','peak_paged','paged','peak_nonpaged','nonpaged','pagefile','peak_pagefile']]
    pm=PMC(); pm.cb=ctypes.sizeof(pm)
    kernel=ctypes.WinDLL('kernel32',use_last_error=True)
    kernel.GetCurrentProcess.restype=ctypes.c_void_p
    psapi=ctypes.WinDLL('psapi',use_last_error=True)
    psapi.GetProcessMemoryInfo.argtypes=[ctypes.c_void_p,ctypes.c_void_p,ctypes.c_ulong]
    if not psapi.GetProcessMemoryInfo(kernel.GetCurrentProcess(),ctypes.byref(pm),pm.cb):
        raise ctypes.WinError(ctypes.get_last_error())
    return int(pm.ws),int(pm.peak_ws)

def available_ram():
    class MS(ctypes.Structure):
        _fields_=[('length',ctypes.c_ulong),('load',ctypes.c_ulong)]+[(k,ctypes.c_ulonglong) for k in
          ['total','available','totalpage','availablepage','totalvirtual','availablevirtual','extended']]
    ms=MS();ms.length=ctypes.sizeof(ms)
    if not ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(ms)): raise ctypes.WinError()
    return int(ms.available)

def iter_trajectories(path):
    cols=['user_id','trajectory_id','timestamp','observation_segment','latitude','longitude','altitude_m']
    carry=None
    for chunk in pd.read_csv(path,usecols=cols,chunksize=CHUNKSIZE,
            dtype={'user_id':str,'trajectory_id':str,'observation_segment':'int64'}):
        chunk['timestamp']=pd.to_datetime(chunk.timestamp,utc=True,errors='raise')
        if chunk[[c for c in cols if c != 'altitude_m']].isna().any().any(): raise ValueError('Missing cleaned timeline/coordinate')
        if carry is not None: chunk=pd.concat([carry,chunk],ignore_index=True)
        changed=chunk.user_id.ne(chunk.user_id.shift()) | chunk.trajectory_id.ne(chunk.trajectory_id.shift())
        starts=np.flatnonzero(changed.to_numpy())
        for a,b in zip(starts[:-1],starts[1:]): yield chunk.iloc[a:b].copy()
        carry=chunk.iloc[starts[-1]:].copy()
    if carry is not None and len(carry): yield carry


In [ ]:
def quantile_fields(name, values):
    values = np.asarray(values, dtype=float)
    result = {name+'_count':int(len(values))}
    for suffix,q in [('p50',.5),('p90',.9),('p95',.95),('p99',.99)]:
        result[name+'_'+suffix] = float(np.quantile(values,q)) if len(values) else None
    result[name+'_max'] = float(np.max(values)) if len(values) else None
    return result

def clock_blocks(t, valid):
    dt = np.diff(t)
    if not len(dt): return []
    same_kind = np.r_[False,(dt[1:]==dt[:-1]) & (valid[1:]==valid[:-1])]
    cuts = np.r_[np.flatnonzero(~same_kind),len(dt)]
    return [(int(t[a]),int(t[b]),int(dt[a]),int(b-a),bool(valid[a]),int(a),int(b))
            for a,b in zip(cuts[:-1],cuts[1:])]

def noise_residuals(t, latitude, longitude, valid, speed):
    if len(t)<3: return np.empty((0,2)), np.empty(0,dtype=int)
    dt=np.diff(t); radius=6_371_008.8
    lat=np.radians(latitude);lon=np.radians(longitude)
    wrap=lambda x:(x+np.pi)%(2*np.pi)-np.pi
    east_prev=wrap(lon[:-2]-lon[1:-1])*np.cos(lat[1:-1])*radius
    east_next=wrap(lon[2:]-lon[1:-1])*np.cos(lat[1:-1])*radius
    north_prev=(lat[:-2]-lat[1:-1])*radius
    north_next=(lat[2:]-lat[1:-1])*radius
    prev=np.column_stack([-east_prev,-north_prev]);nxt=np.column_stack([east_next,north_next])
    denom=np.linalg.norm(prev,axis=1)*np.linalg.norm(nxt,axis=1)
    cos_turn=np.divide((prev*nxt).sum(axis=1),denom,out=np.full(len(denom),-1.),where=denom>0)
    mask=valid[:-1]&valid[1:]&(dt[:-1]<=30)&(dt[1:]<=30)
    mask&=(speed[:-1]>=2)&(speed[1:]>=2)
    mask&=np.abs(speed[:-1]-speed[1:])<=.25*np.maximum(speed[:-1],speed[1:])
    mask&=cos_turn>=np.cos(np.radians(20))
    alpha=dt[:-1]/(dt[:-1]+dt[1:])
    residual=-np.column_stack([(1-alpha)*east_prev+alpha*east_next,
                               (1-alpha)*north_prev+alpha*north_next])
    return residual[mask], np.flatnonzero(mask)+1

def update_lag(state, residual, indices):
    adjacent=np.diff(indices)==1
    for axis in range(2):
        x=residual[:-1,axis][adjacent];y=residual[1:,axis][adjacent]
        state[axis]+=np.array([len(x),x.sum(),y.sum(),(x*x).sum(),(y*y).sum(),(x*y).sum()])

def lag_correlation(moment):
    n,sx,sy,sxx,syy,sxy=moment
    denom=(n*sxx-sx*sx)*(n*syy-sy*sy)
    return float(np.clip((n*sxy-sx*sy)/np.sqrt(denom),-1,1)) if n>=2 and denom>0 else None

def moving_segments(t, lat, lon, valid, speed):
    if len(t)<2: return [],0
    run_starts=np.flatnonzero(np.r_[True,~valid])
    run_ends=np.r_[run_starts[1:]-1,len(t)-1]
    selected=[];examined=0
    for first,last in zip(run_starts,run_ends):
        cursor=int(first)
        while cursor<last:
            end=int(np.searchsorted(t,t[cursor]+600,side='left'))
            if end>last: break
            examined+=1;elapsed=np.diff(t[cursor:end+1]);v=speed[cursor:end]
            low=v<.5;bound=np.r_[0,np.flatnonzero(low[1:]!=low[:-1])+1,len(low)]
            longest=max([int(elapsed[a:b].sum()) for a,b in zip(bound[:-1],bound[1:]) if low[a]],default=0)
            share=float((v>=2).mean())
            if share>=.9 and longest<=30:
                selected.append({'start_utc':pd.to_datetime(t[cursor],unit='s',utc=True).isoformat(),
                    'end_utc':pd.to_datetime(t[end],unit='s',utc=True).isoformat(),
                    'start_point_index':cursor,'end_point_index':end,'duration_seconds':int(t[end]-t[cursor]),
                    'start_latitude':float(lat[cursor]),'start_longitude':float(lon[cursor]),
                    'end_latitude':float(lat[end]),'end_longitude':float(lon[end]),
                    'fast_pair_fraction':share,'longest_slow_run_seconds':longest,
                    'selection_rule':'sustained_speed','label_time_basis':'unverified_not_used'})
            cursor=end
    return selected,examined

def summarize_profile(user, state):
    area=CoverageArea(state['runs']);support=CoverageArea(state['supports'])
    dt=np.concatenate(state['dt']) if state['dt'] else np.empty(0,dtype=int)
    observed=np.concatenate(state['valid']) if state['valid'] else np.empty(0,dtype=bool)
    v=np.concatenate(state['speeds']) if state['speeds'] else np.empty(0)
    noise=np.concatenate(state['noise']) if state['noise'] else np.empty((0,2))
    obs=int(area.cum[-1]);span=int(support.cum[-1])
    row={'user_id':user,'processed_points':state['points'],'processed_trajectories':state['trajectories'],
         'observed_hours':obs/3600,'support_hours':span/3600,'unobserved_support_hours':(span-obs)/3600,
         'unobserved_support_fraction':(span-obs)/span if span else None,
         'missing_altitude_points':state['missing_altitude'],
         'missing_altitude_fraction':state['missing_altitude']/state['points'] if state['points'] else None,
         'short_pairs_excluded_by_segment':int(((dt<=300)&~observed).sum()),
         'speed_gt100mps_pairs':int((v>100).sum()),
         'source_segments_examined':state['examined'],'source_segments':state['source_segments'],
         'has_moving_source':state['source_segments']>0,
         'noise_proxy_status':'available_contaminated_proxy' if len(noise)>=20 else 'insufficient_proxy_samples',
         'noise_proxy_lag1_x':lag_correlation(state['lag'][0]),
         'noise_proxy_lag1_y':lag_correlation(state['lag'][1])}
    row.update(quantile_fields('dt_positive_seconds',dt))
    row.update(quantile_fields('dt_observed_seconds',dt[observed]))
    row.update(quantile_fields('gap_seconds',dt[dt>300]))
    row.update(quantile_fields('speed_mps',v))
    row.update(quantile_fields('noise_proxy_m',np.linalg.norm(noise,axis=1)))
    for minutes in [5,15,60]:
        iv=merge_intervals(state['gaps'][minutes])
        missing=sum((e-s)-int(area.between(s,e)) for s,e in iv)
        row[f'gap_gt_{minutes}m_pairs']=len(state['gaps'][minutes])
        row[f'gap_gt_{minutes}m_unobserved_union_hours']=missing/3600
    distribution=[]
    for flag,label in [(True,'observed_pair'),(False,'unobserved_or_segment_break')]:
        values,counts=np.unique(dt[observed==flag],return_counts=True)
        distribution.extend({'user_id':user,'pair_kind':label,'dt_seconds':int(x),'count':int(n)} for x,n in zip(values,counts))
    return row,distribution


## 3. Lượt quét profile toàn dataset — bạn chủ động chạy
Bật `RUN_B1_SCAN=True`, kiểm tra compute đã dùng trong `COMPUTE_USED_BEFORE_SECONDS`, rồi chạy
các cells theo thứ tự. Trần compute mới tích lũy 24 giờ, mỗi lượt tối đa 12 giờ; không có giờ công.
Output dưới `profiles_v1`: `profiles_per_user.csv`, `sampling_blocks.csv.gz`,
`sampling_distribution.csv`, `moving_segments.csv`, `profile_audit.json`.
Chỉ dùng kết quả khi audit completed và đủ scope; failed/partial không thành full-data.


In [ ]:
def scan_profiles(path, output, inventory, prep_audit, chunksize=100_000,
                  compute_used_before_seconds=0., wall_budget_seconds=12*3600, ram_cap_bytes=None):
    path=Path(path);output=Path(output)
    if output.exists() and any(output.iterdir()):
        raise FileExistsError('Có kết quả profile: xem audit trước, không ghi đè hoặc chạy lại âm thầm.')
    if not 0<=compute_used_before_seconds<24*3600: raise ValueError('Compute còn lại phải dương trong trần24 giờ')
    limit=min(float(wall_budget_seconds),12*3600,24*3600-compute_used_before_seconds)
    if limit<0: raise ValueError('Wall budget không được âm')
    expected=sorted(inventory.user_id.astype(str).tolist());baseline=inventory.set_index('user_id')
    initial=path.stat();output.mkdir(parents=True,exist_ok=True)
    audit={'status':'running','scope':'full_dataset','scanner_revision':'b1-profiles-v1',
        'expected_users':expected,'expected_points':int(prep_audit['totals']['output_points']),
        'expected_trajectories':int(prep_audit['expected_trajectory_files']),
        'dataset_fingerprint':prep_audit.get('dataset_fingerprint'),'completed_users':[],
        'processed_points':0,'processed_trajectories':0,'counts_cover_completed_users_only':True,
        'cleaned_bytes':initial.st_size,'cleaned_mtime_ns':initial.st_mtime_ns,
        'compute_used_before_seconds':compute_used_before_seconds,'wall_budget_seconds':limit,
        'noise_proxy_rule':'linear-neighbor residual; dt<=30s; speed>=2m/s; speed diff<=25%; turn<=20deg',
        'source_rule':'speed blocks >=600s; fast-pair fraction>=90%; longest <0.5m/s run<=30s',
        'label_time_basis':'unverified_not_used_for_source','note':'No generator/detector/inference; proxy is not GPS error truth.'}
    started=time.perf_counter();rows=[];seen=set();current=None;state=None;trajectories=set()
    cap=min(8*1024**3,available_ram()//2) if ram_cap_bytes is None else int(ram_cap_bytes)
    audit['ram_cap_bytes']=cap
    audit['ram_note']='Working-set checkpoint checks, not a hard OS limit. Peak includes kernel lifetime.'
    writej(output/'profile_audit.json',audit)
    def budget():
        ws,peak=process_memory();audit['peak_working_set_bytes']=peak
        if ws>cap: raise MemoryError(f'Working set {ws} exceeds {cap}')
        if time.perf_counter()-started>limit: raise TimeoutError('Profile scan vượt compute budget')
    def finish():
        budget();row,distribution=summarize_profile(current,state)
        ref=baseline.loc[current]
        if row['processed_points']!=int(ref.output_points): raise ValueError('Point-count parity')
        if row['processed_trajectories']!=int(ref.trajectory_files): raise ValueError('Trajectory-count parity')
        if abs(row['observed_hours']*60-float(ref.observed_minutes))>=1e-5: raise ValueError('Coverage parity')
        row['scan_elapsed_seconds']=time.perf_counter()-state['started'];budget()
        rows.append(row);dist_writer.writerows(distribution)
        pd.DataFrame(rows).to_csv(output/'profiles_per_user.csv',index=False)
        audit['completed_users'].append(current);audit['processed_points']+=state['points']
        audit['processed_trajectories']+=state['trajectories'];audit['elapsed_seconds']=time.perf_counter()-started
        writej(output/'profile_audit.json',audit)
        print(current,'points=',state['points'],'moving sources=',state['source_segments'],'noise proxy n=',row['noise_proxy_m_count'])
    try:
        with ExitStack() as stack:
            clock_file=stack.enter_context(gzip.open(output/'sampling_blocks.csv.gz','wt',encoding='utf-8',newline=''))
            clock_writer=csv.writer(clock_file)
            clock_writer.writerow(['user_id','trajectory_id','start_epoch_seconds','end_epoch_seconds',
                'dt_seconds','repetitions','observed_pair','start_pair_index','end_pair_index_exclusive'])
            source_file=stack.enter_context((output/'moving_segments.csv').open('w',encoding='utf-8',newline=''))
            source_cols=['user_id','trajectory_id','start_utc','end_utc','start_point_index','end_point_index',
                'duration_seconds','start_latitude','start_longitude','end_latitude','end_longitude',
                'fast_pair_fraction','longest_slow_run_seconds','selection_rule','label_time_basis']
            source_writer=csv.DictWriter(source_file,fieldnames=source_cols);source_writer.writeheader()
            dist_file=stack.enter_context((output/'sampling_distribution.csv').open('w',encoding='utf-8',newline=''))
            dist_writer=csv.DictWriter(dist_file,fieldnames=['user_id','pair_kind','dt_seconds','count']);dist_writer.writeheader()
            global CHUNKSIZE
            previous_chunksize=CHUNKSIZE;CHUNKSIZE=chunksize
            try:
                for traj in iter_trajectories(path):
                    budget();user=str(traj.user_id.iloc[0]);key=str(traj.trajectory_id.iloc[0])
                    if user!=current:
                        if current is not None: finish();seen.add(current)
                        if user in seen or user not in expected: raise ValueError('Invalid/noncontiguous user')
                        current=user;trajectories=set()
                        state={'points':0,'trajectories':0,'runs':[],'supports':[],'dt':[],'valid':[],'speeds':[],
                            'noise':[],'lag':np.zeros((2,6)),'gaps':{5:[],15:[],60:[]},'missing_altitude':0,
                            'examined':0,'source_segments':0,'started':time.perf_counter()}
                    if key in trajectories: raise ValueError('Noncontiguous trajectory')
                    trajectories.add(key);ns=traj.timestamp.astype('int64').to_numpy()
                    if np.any(ns%10**9): raise ValueError('Timestamp subsecond precision unexpected; do not truncate')
                    t=ns//10**9;seg=traj.observation_segment.to_numpy();dt=np.diff(t)
                    if np.any(dt<=0) or np.any(np.diff(seg)<0): raise ValueError('Invalid cleaned timeline order')
                    state['points']+=len(traj);state['trajectories']+=1
                    state['missing_altitude']+=int(traj.altitude_m.isna().sum())
                    if len(t)<2: continue
                    valid=(dt<=300)&(seg[1:]==seg[:-1]);state['dt'].append(dt);state['valid'].append(valid)
                    state['supports'].append([int(t[0]),int(t[-1])])
                    cuts=np.flatnonzero(np.r_[True,~valid]);ends=np.r_[cuts[1:]-1,len(t)-1]
                    state['runs'].extend([[int(t[a]),int(t[b])] for a,b in zip(cuts,ends) if b>a])
                    clock_writer.writerows((user,key,*block) for block in clock_blocks(t,valid))
                    lat=traj.latitude.to_numpy();lon=traj.longitude.to_numpy()
                    speed=haversine_m(lat[:-1],lon[:-1],lat[1:],lon[1:])/dt
                    state['speeds'].append(speed[valid])
                    residual,indices=noise_residuals(t,lat,lon,valid,speed)
                    state['noise'].append(residual);update_lag(state['lag'],residual,indices)
                    sources,examined=moving_segments(t,lat,lon,valid,speed)
                    source_writer.writerows(dict(user_id=user,trajectory_id=key,**x) for x in sources)
                    state['examined']+=examined;state['source_segments']+=len(sources)
                    for minutes in state['gaps']:
                        ix=np.flatnonzero(dt>minutes*60)
                        state['gaps'][minutes].extend([[int(t[i]),int(t[i+1])] for i in ix])
                if current is not None: finish()
            finally:
                CHUNKSIZE=previous_chunksize
        if sorted(audit['completed_users'])!=expected: raise ValueError('Incomplete users')
        if audit['processed_points']!=audit['expected_points']: raise ValueError('Incomplete points')
        if audit['processed_trajectories']!=audit['expected_trajectories']: raise ValueError('Incomplete trajectories')
        final=path.stat()
        if (final.st_size,final.st_mtime_ns)!=(initial.st_size,initial.st_mtime_ns): raise ValueError('Input changed')
        budget();audit['status']='completed';audit['elapsed_seconds']=time.perf_counter()-started
        audit['compute_used_after_seconds']=compute_used_before_seconds+audit['elapsed_seconds']
        audit['users_with_moving_source']=sum(x['has_moving_source'] for x in rows)
        audit['users_with_noise_proxy']=sum(x['noise_proxy_status']=='available_contaminated_proxy' for x in rows)
        writej(output/'profile_audit.json',audit)
        return audit
    except BaseException as error:
        audit.update(status='failed',active_user=current,error=f'{type(error).__name__}: {error}',
                     elapsed_seconds=time.perf_counter()-started)
        audit['compute_used_after_seconds']=compute_used_before_seconds+audit['elapsed_seconds']
        writej(output/'profile_audit.json',audit)
        raise


In [ ]:
if RUN_B1_SCAN:
    profile_result=scan_profiles(CLEANED,PROFILE_OUT,users,prep,CHUNKSIZE,
        compute_used_before_seconds=COMPUTE_USED_BEFORE_SECONDS,
        wall_budget_seconds=WALL_BUDGET_SECONDS)
else:
    print('Chưa chạy scan profile. Bật RUN_B1_SCAN=True khi sẵn sàng; luôn đọc toàn dataset, không chọn user subset.')


## 4. Đọc kết quả và dependency tiếp theo
Chỉ khi audit completed và đủ users/trajectories/points mới dùng profile cho B2. Bảng nguồn và số
user có labels/≥5 baseline stays ở đây chưa là phép chia chọn/kiểm tra; B2 sẽ khóa seed và cohort.
Nếu bất kỳ nhánh (c) được biết trước thiếu cohort, B4 chỉ mô tả, không chọn ứng viên/không xác nhận.
Ổn định đo mức detector, không chứng minh radius cụ thể. Duration 5/20 phút là trade-off; full-data
cho hai cấu hình này chỉ khi còn compute sau các việc bắt buộc. Không có lịch hay giờ công.


In [ ]:
audit_path=PROFILE_OUT/'profile_audit.json'
if not audit_path.exists():
    print('Chưa có profile scan theo thiết kế mới. Inventory và kết quả thiết kế cũ không thay thế nó.')
else:
    result=json.loads(audit_path.read_text(encoding='utf-8'))
    print('Profile scan:',result['status'])
    if result['status']=='completed' and result.get('scanner_revision')=='b1-profiles-v1':
        profiles=pd.read_csv(PROFILE_OUT/'profiles_per_user.csv',dtype={'user_id':str})
        assert set(profiles.user_id)==set(expected_users)
        assert profiles.processed_points.sum()==prep['totals']['output_points']
        joined=profiles.merge(users[['user_id','label_rows','stay_count']],on='user_id',validate='one_to_one')
        source=joined.has_moving_source
        print('Users có nguồn di chuyển:',int(source.sum()))
        print('Có nguồn + label file (chưa đủ nghĩa có nhãn dùng được):',int((source&(joined.label_rows>0)).sum()))
        print('Có nguồn + >=5 baseline stays:',int((source&(joined.stay_count>=5)).sum()))
        print('Giao cả ba:',int((source&(joined.label_rows>0)&(joined.stay_count>=5)).sum()))
        print('Unobserved support hours, không phải dwell:',profiles.unobserved_support_hours.sum())
        display(profiles)
    else:
        print('Chưa có kết quả completed đúng phiên bản; không suy luận từ các user đã chạy xong.')


**Ghi chú B1 lịch sử — đã đóng theo report ngày 2026-11-04. Không chạy lại lệnh CLI dưới đây.**

## 5. B1 còn phải đối chiếu và lệnh CLI riêng
Raw duplicate khác altitude/cross-file conflicts, label timezone/modes mâu thuẫn và IANA inventory
vẫn cần rà riêng; scanner cleaned không thể khôi phục raw đã gộp. CLI heuristic/parity và timing
inference còn thiếu, runtime scanner không thay runtime detector. Chưa đóng B1 khi dependency thiếu.

Lệnh dưới **chưa tự thực thi**, dùng đúng Python (GeoLife) và output riêng khi bạn chủ động chạy:
```powershell
$env:VIRTUAL_ENV = (Resolve-Path .venv).Path
py -X utf8 -m geolife_inference infer --dataset "data/Geolife Trajectories 1.3" --model heuristic-v1 --config configs/baseline.toml --output artifacts/pipeline/heuristic-v1 --audit-points
```
B1 → B2 → B3 → B4 → B5 → B6. CP2 chỉ bắt đầu khi dependency/job thực sự hoàn tất.
Giữ vote20 trừ lỗi audit cụ thể; chỉ đổi detector/radius khi qua protocol đã khóa và xác nhận cuối,
hoặc giữ baseline trong chế độ mô tả. Không suy F1 tổng hợp thành accuracy trên GeoLife thật.
Nếu không có bằng chứng phân biệt: **100 m/10 phút chưa được chứng minh tối ưu**.
